# Training a Sentiment Classifier

This is a notebook for [CMU CS11-711 Advanced NLP](http://phontron.com/class/anlp2024/) that trains a sentiment classifier based on data. Specifically, it uses a bag-of-words to extract features, and the structured perceptron algorithm to train the classifier.

It will take in a text `X` and return a `label` of "1" if the sentiment of the text is positive, "-1" if the sentiment of the text is negative, and "0" if the sentiment of the text is neutral. You can test the accuracy of your classifier on the [Stanford Sentiment Treebank](http://nlp.stanford.edu/sentiment/index.html) by running the notebook all the way to end.

## Setup

Setup code, do imports.

In [1]:
import random
import tqdm
from math import log2
from collections import defaultdict

import nltk
import pandas as pd
import numpy as np


from nltk.stem.wordnet import WordNetLemmatizer
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

## Feature Extraction

Feature extraction code, how do we get the features we use in training? By default we just use every word.

In [2]:
def extract_features(x: str) -> dict[str, float]:
    features = {}
    lemmatizer = PorterStemmer().stem
    x = word_tokenize(x.lower())
    x = list(map(lemmatizer, x))
    x_split = x
    for x in x_split:
        features[x] = 1
    return features

Also, initialize the feature weights to zero.

In [3]:

feature_weights = {}

## Data Reading

Read in the data from the training and dev (or finally test) sets

In [4]:
def read_xy_data(filename: str) -> tuple[list[str], list[int]]:
    x_data = []
    y_data = []
    with open(filename, 'r') as f:
        for line in f:
            label, text = line.strip().split(' ||| ')
            x_data.append(text)
            y_data.append(int(label))
    return x_data, y_data

In [5]:
x_train, y_train = read_xy_data('../data/sst-sentiment-text-threeclass/train.txt')
x_dev, y_dev = read_xy_data('../data/sst-sentiment-text-threeclass/dev.txt')

In [6]:
print(x_train[0])
print(y_train[0])

The Rock is destined to be the 21st Century 's new `` Conan '' and that he 's going to make a splash even greater than Arnold Schwarzenegger , Jean-Claud Van Damme or Steven Segal .
1


In [7]:
train_df = pd.DataFrame(
    data={"text": x_train, "topic": y_train}
)

In [8]:
probs = {
    "total": {
        "docs": len(train_df),
    }
}

for topic, df in train_df.groupby("topic"):
    probs[topic] = {}
    probs[topic]["docs"] = len(df)
    probs[topic]["proba"] = len(df) / probs["total"]["docs"]
    probs[topic]["words"] = {}
    for text in tqdm.tqdm(df.text):
        bow = extract_features(text)
        for token in bow:
            if token not in probs[topic]["words"]:
                probs[topic]["words"][token] = 0
            probs[topic]["words"][token] += 1 / len(df)
    p_mean = 0
    for token, prob in probs[topic]["words"].items():
        p_mean += prob
    p_mean /= len(probs[topic]["words"])
    probs[topic]["p_mean"] = p_mean
    


100%|██████████| 3610/3610 [00:00<00:00, 6750.02it/s]


In [10]:
def naive_bayes_classifier(text: str, context: dict, non_verbose=True) -> int:
    topic_proba = {}
    for topic in [-1, 0, 1]:
        acc = log2(context[topic]["proba"])
        for word in extract_features(text):
            acc += log2(context[topic]["words"].get(word, 1 / 16000))
        topic_proba[topic] = acc
    total_proba = 2 ** (topic_proba[-1]) + 2 ** (topic_proba[0]) + 2 ** (topic_proba[1])
    res_proba = {
        -1: (2 ** (topic_proba[-1])) / total_proba,
        0: (2 ** (topic_proba[0])) / total_proba,
        1: (2 ** (topic_proba[1])) / total_proba,
    }
    if not non_verbose:
        print(text)
        print(res_proba)
    max_val, max_topic = None, None
    for topic, val in res_proba.items():
        if max_val is None:
            max_val, max_topic = val, topic
        elif val > max_val:
            max_val = val
            max_topic = topic
    return max_topic

## Inference Code

How we run the classifier.

In [11]:
eps = 0.05

def run_classifier(features: dict[str, float]) -> int:
    score = 0
    for feat_name, feat_value in features.items():
        score = score + feat_value * feature_weights.get(feat_name, 0)
    if score > eps:
        return 1
    elif score < -eps:
        return -1
    else:
        return 0

## Training Code

Learn the weights of the classifier.

In [12]:
LR = 0.01

NUM_EPOCHS = 5
for epoch in range(1, NUM_EPOCHS+1):
    # Shuffle the order of the data
    data_ids = list(range(len(x_train)))
    random.shuffle(data_ids)
    # Run over all data points
    for data_id in tqdm.tqdm(data_ids, desc=f'Epoch {epoch}'):
        x = x_train[data_id]
        y = y_train[data_id]
        # We will skip neutral examples
        if y == 0:
            continue
        # Make a prediction
        features = extract_features(x)
        predicted_y = run_classifier(features)
        # Update the weights if the prediction is wrong
        if predicted_y != y:
            for feature in features:
                feature_weights[feature] = feature_weights.get(feature, 0) + y * features[feature] * LR

Epoch 5: 100%|██████████| 8544/8544 [00:01<00:00, 7993.46it/s]


## Evaluation Code

How we evaluate the classifier:

In [13]:
def calculate_accuracy(x_data: list[str], y_data: list[int]) -> float:
    total_number = 0
    correct_number = 0
    for x, y in zip(x_data, y_data):
        y_pred = naive_bayes_classifier(x, probs)
        total_number += 1
        if y == y_pred:
            correct_number += 1
    return correct_number / float(total_number)

In [14]:
label_count = {}
for y in y_dev:
    if y not in label_count:
        label_count[y] = 0
    label_count[y] += 1
print(label_count)

{1: 444, 0: 229, -1: 428}


In [15]:
train_accuracy = calculate_accuracy(x_train, y_train)
test_accuracy = calculate_accuracy(x_dev, y_dev)
print(f'Train accuracy: {train_accuracy}')
print(f'Dev/test accuracy: {test_accuracy}')

Train accuracy: 0.8906835205992509
Dev/test accuracy: 0.59763851044505


## Error Analysis

An important part of improving any system is figuring out where it goes wrong. The following two functions allow you to randomly observe some mistaken examples, which may help you improve the classifier. Feel free to write more sophisticated methods for error analysis as well.

In [16]:
def find_errors(x_data, y_data):
    error_ids = []
    y_preds = []
    for i, (x, y) in enumerate(zip(x_data, y_data)):
        y_preds.append(naive_bayes_classifier(x, probs, True))
        if y != y_preds[-1]:
            error_ids.append(i)
    for _ in range(5):
        my_id = random.choice(error_ids)
        x, y, y_pred = x_data[my_id], y_data[my_id], y_preds[my_id]
        print(f'{x}\ntrue label: {y}\npredicted label: {y_pred}\n')

In [17]:
find_errors(x_dev, y_dev)

But it still jingles in the pocket .
true label: 1
predicted label: -1

A perplexing example of promise unfulfilled , despite many charming moments .
true label: 0
predicted label: 1

Smith is careful not to make fun of these curious owners of architectural oddities .
true label: 0
predicted label: 1

Generally provides its target audience of youngsters enough stimulating eye and ear candy to make its moral medicine go down .
true label: 0
predicted label: 1

It 's slow -- very , very slow .
true label: -1
predicted label: 1



In [18]:
naive_bayes_classifier(
    "Remember the kind of movie we were hoping `` Ecks vs. Sever '' or `` xXx '' was going to be ?",
    probs
)

-1